# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [ ]:
'''
Source: docs/flyrank-seo-research-march-2026.pdf — "The State of AI-Driven SEO," March 2026.

### Finding A — ML Appendix, "What Predicts Health?" (Random Forest feature importance)

The paper reports Random Forest importance for predicting health_score: Average Position 43%,
Impressions 32%, Scroll Depth 15%, CTR 8% — together 98% of total importance.

Where the label comes from: health_score is explicitly defined in the paper's own Methodology
section as Impressions (30 pts) + Position (30 pts) + CTR (20 pts) + Scroll Depth (20 pts).
That's not an independently observed outcome — it's a formula built directly from four inputs.

Does the validation design carry the claim? Not really, and to the paper's credit it says so
itself: "the target itself is partly constructed from some of these inputs, so importance is
descriptive rather than causal." But it's worth going one step further — this is Leakage
Taxonomy pattern #1 from the hunting-leakage-and-validating skill almost exactly: "the label
was computed FROM a column, and that column is in the features." An 80/20 holdout split fixes
overfitting to noise, but it can't fix a label built from its own top predictors — no split
design solves that, only removing the circular features would. Constructive suggestion: re-run
the same importance ranking with avg_position, impressions, ctr, and scroll_depth excluded —
whatever ranks highest among the remaining features would be the genuinely new information,
closer to the skill's "train-without-the-suspect" confession test.

### Finding B — ML Appendix, "What Predicts Growth?" (Logistic Regression, 71% holdout accuracy)

The paper reports 71% holdout accuracy separating growing from declining pages, with Content
Age, Days Since Update, and Days Visible as the strongest coefficients.

Where the label comes from: Trend Direction, defined earlier as the 30-day vs. previous-30-day
impression change — the same construction as is_declining_label in this internship's dataset.

Does the validation design carry the claim? Two questions:
1. Base rate. The portfolio-wide growing/declining split is roughly 62%/38% (Finding #1: 74.8K
   up vs. 45.6K down). A "predict majority class" rule already clears ~62%. 71% is real, but
   it's about 9 points of lift over that floor, not 71 points of skill — the paper doesn't
   print the base rate next to the accuracy number.
2. Split type. The methodology section reports "80/20 split" with no mention of grouping by
   brand, across a dataset spanning 57 brands. If that's a plain random row split, pages from
   the same brand could land in both train and test, and brand-level patterns could leak across
   the split. Constructive suggestion: report accuracy next to the base rate, and re-run once
   with a brand-grouped split alongside the existing number.

'''

In [ ]:
# Backing Finding B's base-rate arithmetic with the paper's own printed numbers
# (Finding #1: up 74.8K vs down 45.6K)
up, down = 74.8e3, 45.6e3
base_rate_majority_class = up / (up + down)
print("Paper's portfolio-wide growing/declining split -> majority-class base rate:",
      round(base_rate_majority_class, 3))
print("Paper's reported holdout accuracy: 0.71")
print("Real lift over the majority-class baseline:", round(0.71 - base_rate_majority_class, 3),
      "points, not 71 points")

# For comparison: our own lane's base rate on the starter CSV
import pandas as pd
df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
own_base_rate = df["trend_direction"].str.lower().eq("down").mean()
print("\nFor reference, our own (much smaller) dataset's decline base rate:", round(own_base_rate, 3))

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [ ]:
'''
ML-08 already used a client-grouped split for the reasons in that notebook (single snapshot,
no time dimension; grouping avoids same-client rows leaking across train/test). Here I rebuild
the same feature vector and Random Forest, then compare a plain random row-level split
("before" — the naive default) against the same client-grouped split ("after" — what ML-08
actually used), on the same data and metric (precision@50), to see how large the gap really is.

'''

In [ ]:
import numpy as np
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.ensemble import RandomForestClassifier

RANDOM_SEED = 42
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

fv = pd.DataFrame(index=df.index)
fv["content_id"] = df["content_id"]
fv["client_id"] = df["client_id"]
for col in ["impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d"]:
    fv[f"log_{col}"] = np.log1p(df[col])
for col in ["content_age_days", "days_since_last_update", "ctr", "avg_position",
            "engagement_rate", "ai_traffic_pct", "days_with_impressions", "days_with_sessions"]:
    fv[col] = df[col]
fv["has_scroll_rate"] = df["scroll_rate"].notna().astype(int)
fv["scroll_rate"] = df["scroll_rate"].fillna(0)
fv["has_word_count"] = df["word_count"].notna().astype(int)
fv["word_count_filled"] = df["word_count"].fillna(0)
fv["has_keyword_data"] = df["search_volume"].notna().astype(int)
fv["search_volume_filled"] = df["search_volume"].fillna(0)
fv["competition_filled"] = df["competition"].fillna(0)
fv["content_type"] = df["content_type"]
fv["main_intent"] = df["main_intent"].fillna("unknown")
fv["competition_level"] = df["competition_level"].fillna("unknown")
fv = pd.get_dummies(fv, columns=["content_type", "main_intent", "competition_level"],
                     prefix=["ctype", "intent", "complevel"])
fv["is_declining_label"] = df["is_declining_label"]

feature_cols = [c for c in fv.columns if c not in ("content_id", "client_id", "is_declining_label")]
X = fv[feature_cols]
y = fv["is_declining_label"]
groups = fv["client_id"]

def precision_at_k(scores, labels, k=50):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

def fit_and_score(X_train, X_test, y_train, y_test):
    model = RandomForestClassifier(n_estimators=300, max_depth=8, random_state=RANDOM_SEED, n_jobs=-1)
    model.fit(X_train, y_train)
    scores = model.predict_proba(X_test)[:, 1]
    return precision_at_k(scores, y_test.values)

# BEFORE: plain random row-level split
Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_SEED, stratify=y)
random_p50 = fit_and_score(Xr_train, Xr_test, yr_train, yr_test)

# AFTER: client-grouped split (what ML-08 actually used)
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_SEED)
train_idx, test_idx = next(gss.split(X, y, groups=groups))
grouped_p50 = fit_and_score(X.iloc[train_idx], X.iloc[test_idx], y.iloc[train_idx], y.iloc[test_idx])

print(f"BEFORE (random row split):   precision@50 = {random_p50:.3f}")
print(f"AFTER  (client-grouped split): precision@50 = {grouped_p50:.3f}")
print(f"Gap: {random_p50 - grouped_p50:.3f}")

In [ ]:
'''
The gap is real and large: 0.900 -> 0.720, an 18-point drop. The random split let the model
partly memorize client-level patterns — even though no single feature is client-identifying,
client_id correlates with things like typical content length, publishing cadence, and CMS
quirks that repeat within a client's pages. A random split puts some of each client's pages in
both train and test, so the model gets a preview of that client's "style" before being tested
on it. The client-grouped number (0.72, matching what ML-08 already reported) is the honest
one. This is exactly the finding the hunting-leakage-and-validating skill predicts: "the GAP
between them is itself a finding about how much memorization was happening." Good news: ML-08
already used the grouped split, so nothing already submitted needs correcting — this just
proves why that choice mattered.

'''

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [ ]:
# Same hunt as ML-05, run against the ACTUAL final feature_cols used to train the ML-08 model
# (post one-hot-encoding), not just the pre-encoding column names.

LABEL_DERIVED = ["trend_direction", "trend_pct",
                  "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
                  "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d"]
PRODUCT_FLAGS = ["provider_used", "model_used"]

leaked_label = [c for c in feature_cols if c in LABEL_DERIVED]
leaked_product = [c for c in feature_cols if c in PRODUCT_FLAGS]
print("Leakage test 1 (label-derived columns in final feature_cols):",
      "PASS — none present" if not leaked_label else f"FAIL: {leaked_label}")
print("Leakage test 2 (product flags in final feature_cols):",
      "PASS — none present" if not leaked_product else f"FAIL: {leaked_product}")

# Confession test: add trend_pct back in and watch the score jump
X_with_leak = X.copy()
X_with_leak["trend_pct_LEAK"] = df.loc[X_with_leak.index, "trend_pct"].values
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_SEED)
tr_idx, te_idx = next(gss2.split(X_with_leak, y, groups=groups))
leaked_p50 = fit_and_score(X_with_leak.iloc[tr_idx], X_with_leak.iloc[te_idx],
                            y.iloc[tr_idx], y.iloc[te_idx])
print(f"\nConfession test: precision@50 WITH trend_pct deliberately added back in: {leaked_p50:.3f}")
print(f"Compare to the honest grouped-split number without it: {grouped_p50:.3f}")
print("-> if the harness is working, adding the leak should push the score toward 1.0.")

# Top feature importance sanity-check
rf_final = RandomForestClassifier(n_estimators=300, max_depth=8, random_state=RANDOM_SEED, n_jobs=-1)
rf_final.fit(X.iloc[train_idx], y.iloc[train_idx])
top_importance = pd.Series(rf_final.feature_importances_, index=feature_cols).sort_values(ascending=False)
print("\nTop feature:", top_importance.index[0], "at", round(top_importance.iloc[0], 3),
      "importance —", "suspiciously dominant (>0.5), investigate" if top_importance.iloc[0] > 0.5
      else "not dominant, no single feature towers over the rest")

In [ ]:
'''
Attack checklist, all four run:
- Label-derived columns in the final feature_cols: none — clean.
- Product/production-detail flags in the final feature_cols: none — clean.
- Confession test (deliberately adding trend_pct back in): precision@50 jumped from 0.720 to
  1.000 — exactly the "collapse toward 1.0" signature the skill describes, confirming the test
  harness itself is working and would have caught a real leak if one existed.
- Top feature importance: days_with_impressions at 0.182 — not dominant (nothing near the 0.5+
  that would demand investigation, unlike Finding A's Random Forest in Section 1, where the top
  four features accounted for 98% of importance).

Base rate and split, both already covered: test base rate 0.517 (ML-08), reported next to
every metric; client-grouped split confirmed with zero client overlap (ML-08 and Section 2
above). All six items on the skill's attack checklist are accounted for.

'''

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [ ]:
boldest_original = ("A page is worth a CTR-fix review if it already ranks on page 1 but its "
    "click-through rate is below the typical page-1 rate... meaning the ranking effort already "
    "worked, but the title/snippet isn't converting that visibility into clicks.")

claim_ladder_check = {
    "evidence actually have": "A measured comparison (page-1 pages, CTR below vs at/above the page-1 median) "
                               "on one snapshot, no experiment run on the snippet itself",
    "words the original used": "'the ranking effort already worked' / 'isn't converting' "
                                "-> states WHY the CTR is low as settled fact (a causal claim)",
    "words the evidence supports": "'associated with' / 'flags for review' -> decision-support, not causal",
}
for k, v in claim_ladder_check.items():
    print(f"{k}: {v}")

In [ ]:
'''
Rewrite, using the claim ladder from writing-honest-claims:

Original (too bold): "...meaning the ranking effort already worked, but the title/snippet
isn't converting that visibility into clicks."

This states a cause (the title/snippet is the reason) as settled fact, from a single snapshot
with no experiment on the snippet itself — a classic case of the words claiming more than the
evidence measured.

Safe rewrite: "In this snapshot, pages already ranking on page 1 with click-through rates below
the typical page-1 rate are worth flagging for a title/snippet review. This is a
decision-support signal, not a diagnosis — the low CTR is observed and measured against the
page-1 median (Section 1, ML-07), but this snapshot doesn't test whether a snippet rewrite is
the actual fix; a SERP feature capturing the click, seasonal dip, or a mismatch between title
and query intent could produce the same pattern (see the per-row caveats already written in
ML-07's top-20 review)."

What changed: "meaning...worked" (causal) -> "worth flagging for a review" (decision-support).
"Isn't converting" (states the mechanism as fact) -> "observed and measured against..." plus an
explicit list of alternative explanations the data can't rule out.

'''

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.